In [ ]:
from dotenv import load_dotenv
from pathlib import Path

c:\Users\paris\Documents\Cours Centrale 3A\technical-test\injury-prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Database handling
## Nutrition data

To test the food calorie estimation model :

In [ ]:
from utils.calorie_estimation import CalorieEstimation

# Initialize CalorieEstimation using the utility class
model = CalorieEstimation()

# Predict calories for a food image
photo_path = r"data\p01\food-images\IMG_8916.jpeg"
calories = model.predict(photo_path)
print(f"Estimated: {calories:.0f} calories")

[INFO] Downloading/Verifying CalorieCLIP model files locally...


Fetching 19 files: 100%|██████████| 19/19 [00:00<00:00, 142.33it/s]
c:\Users\paris\Documents\Cours Centrale 3A\technical-test\injury-prediction\.venv\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


[INFO] Loading CalorieCLIP into memory on device: cpu...
[INFO] Model loaded successfully!
Estimated: 218 calories


Estimating calories for each player's photos, extracting photo dates, creating individual DataFrames, and concatenating them into a single DataFrame:

In [1]:
from pathlib import Path
from PIL import Image
import pandas as pd
from utils.calorie_estimation import CalorieEstimation

# Initialize CalorieEstimation using the utility class
model = CalorieEstimation()

players = ["p01", "p03", "p05"]
player_dfs = []
BATCH_SIZE = 32

for player in players:
    player_path = Path(f"data/{player}/food-images")
    if not player_path.exists():
        continue
    
    image_paths = []
    for ext in ("*.jpg", "*.jpeg", "*.JPG", "*.JPEG"):
        image_paths.extend(list(player_path.glob(ext)))
    
    if not image_paths:
        continue
        
    dates = []
    calories_list = []
    
    for i in range(0, len(image_paths), BATCH_SIZE):
        batch_paths = image_paths[i:i + BATCH_SIZE]
        calories_batch = model.predict_batch(batch_paths)
        
        for img_path, cals in zip(batch_paths, calories_batch):
            img = Image.open(img_path)
            exif = img.getexif()
            dt = None
            if exif:
                dt_str = exif.get(306)
                if 34665 in exif:
                    sub_exif = exif.get_ifd(34665)
                    dt_str = sub_exif.get(36867, dt_str)
                if dt_str:
                    try:
                        dt = dt_str.split(' ')[0].replace(':', '-')
                    except Exception:
                        dt = dt_str
            if not dt:
                dt = pd.to_datetime(img_path.stat().st_mtime, unit='s').strftime('%Y-%m-%d')
                
            dates.append(dt)
            calories_list.append(float(cals))
            
    df_player = pd.DataFrame({
        'player': player,
        'image': [p.name for p in image_paths],
        'date': dates,
        'calories': calories_list
    })
    player_dfs.append(df_player)

# Concatenate all 3 player DataFrames into 1
combined_df = pd.concat(player_dfs, ignore_index=True)

c:\Users\paris\Documents\Cours Centrale 3A\technical-test\injury-prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INFO] Downloading/Verifying CalorieCLIP model files locally...


Fetching 19 files: 100%|██████████| 19/19 [00:00<00:00, 107.54it/s]


[INFO] Loading CalorieCLIP into memory on device: cpu...


c:\Users\paris\Documents\Cours Centrale 3A\technical-test\injury-prediction\.venv\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


[INFO] Model loaded successfully!


In [2]:
print(combined_df)

     player          image        date    calories
0       p01  IMG_8916.jpeg  2020-02-01  218.451416
1       p01  IMG_8917.jpeg  2020-02-01  447.812347
2       p01  IMG_8918.jpeg  2020-02-01  317.138062
3       p01  IMG_8920.jpeg  2020-02-01  430.066254
4       p01  IMG_8921.jpeg  2020-02-01  203.444107
...     ...            ...         ...         ...
1279    p05   IMG_2924.jpg  2020-03-26  297.093719
1280    p05   IMG_2925.jpg  2020-03-26  240.066147
1281    p05   IMG_2931.jpg  2020-03-27  271.975830
1282    p05   IMG_2966.jpg  2020-03-28  315.789917
1283    p05   IMG_2970.jpg  2020-03-29  578.645020

[1284 rows x 4 columns]
